In [0]:
# ============================================================================
# FULL RESET: Drop all UC tables/schemas and delete ADLS data files
# WARNING: This permanently deletes ALL data in bronze, silver, and gold.
# Re-run Cell 1 + Cell 2 afterwards to re-register empty table shells.
# ============================================================================

CATALOG = "adb_saas_dev"
STORAGE_ACCOUNT = "datalakesaas"

# Step 1: Drop all schemas with CASCADE (drops all tables within)
for schema in ["gold", "silver", "bronze"]:
    spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{schema} CASCADE")
    print(f"✅ Dropped schema: {CATALOG}.{schema}")

# Step 2: Delete underlying ADLS data files
for container in ["bronze", "silver", "gold"]:
    base = f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
    try:
        items = dbutils.fs.ls(base)
        for item in items:
            dbutils.fs.rm(item.path, recurse=True)
            print(f"🗑️  Deleted: {item.path}")
        print(f"✅ Cleaned: {container} container")
    except Exception as e:
        print(f"⚠️  Could not clean {container} files: {e}")

# Step 3: Recreate empty schemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze COMMENT 'Raw landing zone tables'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver COMMENT 'Conformed and cleansed Delta tables'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold COMMENT 'Kimball Star Schema analytics layer'")

print("\n✅ Full reset complete. Empty schemas recreated.")
print("\nNext steps:")
print("  1. Run Cell 1 + Cell 2 to re-register external table shells")
print("  2. Re-ingest bronze data (ADF / external pipeline)")
print("  3. Run 02_transform_silver")
print("  4. Run 03_transform_gold")

In [0]:
%sql
    
-- ============================================================================
-- 0. USE UNITY CATALOG
-- ============================================================================
USE CATALOG adb_saas_dev;

-- ============================================================================
-- 1. CREATE LOGICAL SCHEMAS / DATABASES IN METASTORE
-- ============================================================================
CREATE SCHEMA IF NOT EXISTS silver
COMMENT 'Conformed and cleansed operational Delta Lake tables';

CREATE SCHEMA IF NOT EXISTS gold
COMMENT 'Kimball Star Schema dimensional analytics layer';

-- ============================================================================
-- 2. REGISTER SILVER EXTERNAL TABLES
-- ============================================================================
CREATE TABLE IF NOT EXISTS adb_saas_dev.silver.cln_plans
USING DELTA
LOCATION 'abfss://silver@datalakesaas.dfs.core.windows.net/cln_plans';

CREATE TABLE IF NOT EXISTS adb_saas_dev.silver.cln_organizations
USING DELTA
LOCATION 'abfss://silver@datalakesaas.dfs.core.windows.net/cln_organizations';

CREATE TABLE IF NOT EXISTS adb_saas_dev.silver.cln_users
USING DELTA
LOCATION 'abfss://silver@datalakesaas.dfs.core.windows.net/cln_users';

CREATE TABLE IF NOT EXISTS adb_saas_dev.silver.cln_subscriptions
USING DELTA
LOCATION 'abfss://silver@datalakesaas.dfs.core.windows.net/cln_subscriptions';

CREATE TABLE IF NOT EXISTS adb_saas_dev.silver.cln_invoices
USING DELTA
LOCATION 'abfss://silver@datalakesaas.dfs.core.windows.net/cln_invoices';

CREATE TABLE IF NOT EXISTS adb_saas_dev.silver.cln_feature_usage
USING DELTA
LOCATION 'abfss://silver@datalakesaas.dfs.core.windows.net/cln_feature_usage';

CREATE TABLE IF NOT EXISTS adb_saas_dev.silver.cln_logins
USING DELTA
LOCATION 'abfss://silver@datalakesaas.dfs.core.windows.net/cln_logins';

-- ============================================================================
-- 3. REGISTER GOLD EXTERNAL TABLES (DIMENSIONS & FACTS)
-- ============================================================================
CREATE TABLE IF NOT EXISTS adb_saas_dev.gold.dim_date
USING DELTA
LOCATION 'abfss://gold@datalakesaas.dfs.core.windows.net/dim_date';

CREATE TABLE IF NOT EXISTS adb_saas_dev.gold.dim_plan
USING DELTA
LOCATION 'abfss://gold@datalakesaas.dfs.core.windows.net/dim_plan';

CREATE TABLE IF NOT EXISTS adb_saas_dev.gold.dim_organization
USING DELTA
LOCATION 'abfss://gold@datalakesaas.dfs.core.windows.net/dim_organization';

CREATE TABLE IF NOT EXISTS adb_saas_dev.gold.dim_user
USING DELTA
LOCATION 'abfss://gold@datalakesaas.dfs.core.windows.net/dim_user';

CREATE TABLE IF NOT EXISTS adb_saas_dev.gold.fact_billing
USING DELTA
LOCATION 'abfss://gold@datalakesaas.dfs.core.windows.net/fact_billing';

CREATE TABLE IF NOT EXISTS adb_saas_dev.gold.fact_subscriptions
USING DELTA
LOCATION 'abfss://gold@datalakesaas.dfs.core.windows.net/fact_subscriptions';

CREATE TABLE IF NOT EXISTS adb_saas_dev.gold.fact_feature_usage
USING DELTA
LOCATION 'abfss://gold@datalakesaas.dfs.core.windows.net/fact_feature_usage';

CREATE TABLE IF NOT EXISTS adb_saas_dev.gold.fact_logins
USING DELTA
LOCATION 'abfss://gold@datalakesaas.dfs.core.windows.net/fact_logins';

In [0]:
%sql
-- ============================================================================
-- 1. CREATE BRONZE SCHEMA IN CATALOG
-- ============================================================================
CREATE SCHEMA IF NOT EXISTS bronze
COMMENT 'Raw landing zone tables for MySQL, CSV, and API sources';

-- ============================================================================
-- 2. REGISTER MYSQL OLTP BRONZE TABLES
-- ============================================================================
CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_plans (
    plan_id INT,
    plan_name STRING,
    tier STRING,
    monthly_fee DECIMAL(10,2),
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    _ingested_at TIMESTAMP,
    _source_system STRING,
    ingest_year INT,
    ingest_month INT
)
USING PARQUET
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/sql_plans';

CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_organizations (
    org_id INT,
    name STRING,
    industry STRING,
    country STRING,
    is_active BOOLEAN,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    _ingested_at TIMESTAMP,
    _source_system STRING,
    ingest_year INT,
    ingest_month INT
)
USING PARQUET
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/sql_organizations';

CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_users (
    user_id INT,
    org_id INT,
    email STRING,
    role STRING,
    is_active BOOLEAN,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    _ingested_at TIMESTAMP,
    _source_system STRING,
    ingest_year INT,
    ingest_month INT
)
USING PARQUET
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/sql_users';

CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_subscriptions (
    subscription_id INT,
    org_id INT,
    plan_id INT,
    status STRING,
    started_at TIMESTAMP,
    cancelled_at TIMESTAMP,
    updated_at TIMESTAMP,
    _ingested_at TIMESTAMP,
    _source_system STRING,
    ingest_year INT,
    ingest_month INT
)
USING PARQUET
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/sql_subscriptions';

CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_invoices (
    invoice_id INT,
    subscription_id INT,
    amount_due DECIMAL(10,2),
    amount_paid DECIMAL(10,2),
    status STRING,
    billing_date DATE,
    paid_at TIMESTAMP,
    updated_at TIMESTAMP,
    _ingested_at TIMESTAMP,
    _source_system STRING,
    ingest_year INT,
    ingest_month INT
)
USING PARQUET
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/sql_invoices';

-- ============================================================================
-- 3. REGISTER FILE & API STREAM BRONZE TABLES
-- ============================================================================
CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_feature_usage (
    UserId STRING,
    OrgId STRING,
    UsageDate STRING,
    FeatureName STRING,
    UsageCount STRING,
    _ingested_at STRING,
    _source_file STRING,
    ingest_year INT,
    ingest_month INT
)
USING PARQUET
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/file_feature_usage';

CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_logins (
    _ingested_at STRING,
    _source_file STRING,
    batch_id STRING,
    data ARRAY<STRUCT<client_device: STRING, event_id: STRING, event_timestamp: STRING, ip_address: STRING, login_success: BOOLEAN, user_uid: BIGINT>>,
    has_more BOOLEAN,
    page BIGINT,
    page_size BIGINT,
    record_count BIGINT,
    source STRING,
    status STRING,
    total_count BIGINT,
    ingest_year INT,
    ingest_month INT
)
USING JSON
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/api_logins';

In [0]:
# cln_plans is in UC-managed storage — relocate to datalakesaas ADLS
# Step 1: Copy data to correct ADLS silver path
target_path = "abfss://silver@datalakesaas.dfs.core.windows.net/cln_plans"
df = spark.table("adb_saas_dev.silver.cln_plans")
print(f"cln_plans: {df.count()} rows — copying to ADLS")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(target_path)

# Step 2: Drop managed table and recreate as external at ADLS
# (ALTER TABLE SET LOCATION is not supported on managed tables)
spark.sql("DROP TABLE adb_saas_dev.silver.cln_plans")
spark.sql(f"""
    CREATE TABLE adb_saas_dev.silver.cln_plans
    USING DELTA
    LOCATION '{target_path}'
""")

# Step 3: Verify
detail = spark.sql("DESCRIBE DETAIL adb_saas_dev.silver.cln_plans").select("location").collect()[0]
print(f"\n✅ cln_plans now at: {detail['location']}")